# Iteris — EDA + DRL-Compatibility Diagnostic (CAMUS + BRISC)

One CPU-only report that answers, per class, **"can the contour-refinement agent beat this lite
U-Net baseline, and what's stopping it?"** — so a GPU training round is only spent where it can pay off.

Sections:
- **§3 Dataset EDA** — counts, GT area distribution, topology (connected components), intensity, view/phase.
- **§4 Lite U-Net baseline** — init Dice/IoU distribution, the worst cases (where headroom lives), and
  prob_map calibration (INERT vs USABLE — drives whether the uncertainty gate can work).
- **§5 Error decomposition** — boundary (contour-fixable) vs topology / interior (NOT contour-fixable).
- **§6 DRL compatibility** — contour representational ceiling, GT-oracle ceiling, realistic headroom vs
  the attention competitor, and the gate band fraction.
- **§7 Master table + per-class recommendation** — the actionable summary: which class/agent to push and
  with which knobs (gate on/off, retrain, etc.).

Warm-start runs the lite U-Net once per class (~5 min CAMUS / ~10 min BRISC); everything after is instant.

**Attach (Datasets panel):** `iteris-pkg` (latest), `CAMUS`, `brisc2025`/`BRISC`, and the retrained
lite-U-Net checkpoint dataset (`camus_lite_unet_best.pt` / `brisc_lite_unet_best.pt`).

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached. Add it in the right-hand panel -> Datasets.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'Package at: {PKG_ROOT}')

## 1 · Imports, config, and report helpers

`ATTN_DICE` is the attention U-Net's test Dice per class (the fixed competitor) — used to derive the
*realistic*, non-GT-privileged headroom. `full_report(label)` runs every section below for one class
from its cached warm-start samples and returns a row of summary metrics.

In [ ]:
import numpy as np
import pandas as pd
import scipy.ndimage as ndi

from iteris.config import load_drl_class_config, resolve_agent_config, load_config
from iteris.utils import model_suffix
from iteris.warm_start import precompute_init_masks
from iteris.geometry import dice_score, STRUCT
from iteris.diagnostics import (prob_map_informativeness, error_type_audit,
                                headroom_report, repr_ceiling, oracle_greedy,
                                _base_env_kwargs)

# Attention U-Net competitor test Dice (see CONTEXT.md / SKILLS.md).
ATTN_DICE = {
    'CAMUS_LV_endo': 0.938, 'CAMUS_LV_epi': 0.872, 'CAMUS_LA': 0.896,
    'BRISC_tumor': 0.835,
    'BRISC_glioma': None, 'BRISC_meningioma': None, 'BRISC_pituitary': None,
}

SAMPLES = {}   # label -> dict(train, val, test)
CFGS    = {}   # label -> resolved cfg (geometry source = TD3 block)
RESULTS = {}   # label -> summary-metrics row


def load_class(config_path, root_names, label):
    """Warm-start the lite U-Net for one class; cache (train,val,test) samples."""
    cfg_full = load_drl_class_config(str(PKG_ROOT / 'configs' / config_path))
    cfg = resolve_agent_config(cfg_full, 'TD3')   # TD3 block carries full geometry
    bcfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))
    root = None
    for nm in root_names:
        c = [p for p in Path('/kaggle/input').rglob(nm) if p.is_dir()]
        if c:
            root = str(c[0]); break
    if root is None:
        raise FileNotFoundError(f'none of {root_names} under /kaggle/input')
    bcfg['data_root'] = root
    ck = f"{bcfg['dataset'].lower()}{model_suffix(bcfg)}_best.pt"
    cands = list(Path('/kaggle/input').rglob(ck))
    if not cands:
        raise FileNotFoundError(f'checkpoint {ck!r} not found under /kaggle/input')
    cfg['baseline_checkpoint'] = str(cands[0])
    tr, va, te = precompute_init_masks(
        baseline_cfg=bcfg, baseline_checkpoint=cfg['baseline_checkpoint'],
        target_class=cfg['target_class'], min_area_fraction=cfg.get('min_area_fraction', 0.01),
        tumor_type_filter=cfg.get('tumor_type_filter'))
    SAMPLES[label] = dict(train=tr, val=va, test=te)
    CFGS[label] = cfg
    print(f'  {label}: train {len(tr)} | val {len(va)} | test {len(te)} | ckpt {Path(cfg["baseline_checkpoint"]).name}')
    return cfg


def _entropy(p):
    pc = np.clip(p.astype(np.float32), 1e-6, 1 - 1e-6)
    return float((-(pc*np.log2(pc) + (1-pc)*np.log2(1-pc))).mean())


def full_report(label, n_diag=60):
    """Full EDA + diagnostic for one class. Prints, plots, returns a metrics row."""
    import matplotlib.pyplot as plt
    s = SAMPLES[label]; cfg = CFGS[label]
    val = s['val']
    print('\n' + '=' * 78 + f'\n{label}\n' + '=' * 78)

    # ---- §3 Dataset EDA ----
    area = np.array([float(x['gt_mask'].mean()) for x in val])         # GT area fraction
    ncc  = np.array([ndi.label(x['gt_mask'], structure=STRUCT)[1] for x in val])
    imean = np.array([float(x['image'].mean()) for x in val])
    views, phases = {}, {}
    for x in val:
        views[x.get('view', '')]  = views.get(x.get('view', ''), 0) + 1
        phases[x.get('phase', '')] = phases.get(x.get('phase', ''), 0) + 1
    multi_cc = float((ncc > 1).mean())
    print(f'[dataset] n(train/val/test)={len(s["train"])}/{len(val)}/{len(s["test"])} | '
          f'GT area%: mean {area.mean()*100:.2f} med {np.median(area)*100:.2f} '
          f'[{area.min()*100:.2f},{area.max()*100:.2f}] | multi-blob GT: {multi_cc*100:.1f}% | '
          f'img mean {imean.mean():.3f}')
    if any(views):  print(f'[dataset] views={views} phases={phases}')

    # ---- §4 Lite U-Net baseline ----
    dice = np.array([dice_score(x['init_mask'], x['gt_mask']) for x in val])
    iou  = dice / (2 - dice + 1e-8)
    order = np.argsort(dice)
    worst = order[:5]
    print(f'[lite] init Dice: mean {dice.mean():.4f} med {np.median(dice):.4f} '
          f'p10 {np.percentile(dice,10):.4f} min {dice.min():.4f} | '
          f'IoU mean {iou.mean():.4f} | worst-5 Dice {np.round(dice[worst],3).tolist()}')
    pm = prob_map_informativeness(val, n_samples=n_diag, label=label)
    ent = float(np.mean([_entropy(np.asarray(x['prob_map'])) for x in val[:n_diag]]))
    prob_state = 'USABLE' if pm['uncertain_band_frac'] > 0.01 else 'INERT'
    print(f'[lite] prob_map: band_frac {pm["uncertain_band_frac"]:.4f} | mean entropy {ent:.4f} -> {prob_state}')

    # ---- §5 Error decomposition ----
    et = error_type_audit(val, n_samples=n_diag, label=label)

    # ---- §6 DRL compatibility ----
    hr = headroom_report(
        val, n_points=cfg['n_points'], cont_sectors=cfg.get('cont_sectors', 12),
        disp_px=cfg['disp_px'], spline_smooth=cfg['spline_smooth'],
        n_samples=n_diag, label=label, attention_dice=ATTN_DICE.get(label))

    # ---- plots ----
    fig, ax = plt.subplots(2, 3, figsize=(15, 8))
    ax[0,0].hist(area*100, bins=30, color='C0'); ax[0,0].set(title=f'{label}: GT area %', xlabel='% of image')
    ax[0,1].hist(dice, bins=30, color='C2'); ax[0,1].axvline(dice.mean(), color='k', ls='--')
    ax[0,1].set(title=f'init Dice (mean {dice.mean():.3f})', xlabel='Dice')
    allp = np.concatenate([np.asarray(x['prob_map']).ravel().astype(np.float32) for x in val[:20]])
    ax[0,2].hist(allp, bins=50, color='C3', log=True)
    ax[0,2].axvspan(0.35, 0.65, color='orange', alpha=0.25)
    ax[0,2].set(title=f'prob_map values ({prob_state})', xlabel='p (orange=gate band)')
    ax[1,0].bar([str(k) for k in sorted(set(ncc))], [int((ncc==k).sum()) for k in sorted(set(ncc))], color='C4')
    ax[1,0].set(title='GT connected components', xlabel='# blobs')
    ax[1,1].bar(['boundary','topology','interior'],
                [et['boundary_frac'], et['topology_frac'], et['interior_frac']],
                color=['C2','C3','C1'])
    ax[1,1].set(title='error decomposition', ylim=(0,1))
    bars = ['baseline','repr cap','oracle(GT)']
    vals = [hr['baseline_init_dice'], hr['contour_repr_ceiling'], hr['oracle_greedy_ceiling']]
    if ATTN_DICE.get(label) is not None:
        bars.append('attn'); vals.append(ATTN_DICE[label])
    ax[1,2].bar(bars, vals, color='C0'); ax[1,2].set(title='ceilings vs baseline', ylim=(min(vals)-0.05,1.0))
    ax[1,2].axhline(hr['baseline_init_dice'], color='k', ls='--', lw=0.8)
    plt.suptitle(f'{label} — EDA + diagnostic', fontweight='bold'); plt.tight_layout()
    plt.savefig(f'/kaggle/working/eda_{label}.png', dpi=120); plt.show()

    # qualitative: the 3 worst lite cases (where headroom is)
    fig2, ax2 = plt.subplots(3, 4, figsize=(13, 9))
    for r, idx in enumerate(worst[:3]):
        x = val[int(idx)]
        ax2[r,0].imshow(x['image'], cmap='gray'); ax2[r,0].set_ylabel(f'Dice {dice[idx]:.2f}')
        ax2[r,1].imshow(x['gt_mask'], cmap='Greens'); ax2[r,1].set_title('GT' if r==0 else '')
        ax2[r,2].imshow(x['init_mask'], cmap='Blues'); ax2[r,2].set_title('lite init' if r==0 else '')
        ax2[r,3].imshow(np.asarray(x['prob_map']).astype(np.float32), cmap='magma', vmin=0, vmax=1)
        ax2[r,3].set_title('prob_map' if r==0 else '')
        for c in range(4): ax2[r,c].set_xticks([]); ax2[r,c].set_yticks([])
    plt.suptitle(f'{label} — 3 worst lite cases (largest headroom)', fontweight='bold'); plt.tight_layout()
    plt.savefig(f'/kaggle/working/eda_{label}_worst.png', dpi=120); plt.show()

    realistic = hr.get('realistic_headroom_estimate')
    row = dict(
        label=label, n_val=len(val),
        gt_area_pct=round(area.mean()*100, 2), multi_blob_pct=round(multi_cc*100, 1),
        baseline_dice=round(hr['baseline_init_dice'], 4),
        baseline_dice_p10=round(float(np.percentile(dice,10)), 4),
        prob_state=prob_state, band_frac=round(pm['uncertain_band_frac'], 4),
        boundary_frac=round(et['boundary_frac'], 2),
        topology_interior_frac=round(et['topology_frac']+et['interior_frac'], 2),
        contour_repr_ceiling=round(hr['contour_repr_ceiling'], 4),
        oracle_ceiling_GTpriv=round(hr['oracle_greedy_ceiling'], 4),
        attn_dice=ATTN_DICE.get(label),
        realistic_headroom=(round(realistic, 4) if realistic is not None else None),
        pillar4_go=(pm['uncertain_band_frac'] > 0.01 and et['boundary_frac'] > 0.5),
    )
    RESULTS[label] = row
    return row

## 2 · Warm-start all CAMUS classes
~5 min/class. Each class needs its own U-Net pass (different `target_class`).

In [ ]:
print('Warm-starting CAMUS classes...')
load_class('CAMUS/DRL/camus_drl_c1.yaml', ['CAMUS'], 'CAMUS_LV_endo')
load_class('CAMUS/DRL/camus_drl_c2.yaml', ['CAMUS'], 'CAMUS_LV_epi')
load_class('CAMUS/DRL/camus_drl_c3.yaml', ['CAMUS'], 'CAMUS_LA')

## 3-6 · Per-class EDA + diagnostic report (CAMUS)

In [ ]:
for lab in ['CAMUS_LV_endo', 'CAMUS_LV_epi', 'CAMUS_LA']:
    full_report(lab)

## (optional) BRISC

Off by default — BRISC currently has no realistic headroom (lite baseline already above the attention
competitor). Set `RUN_BRISC = True` to include the generic-tumour class (~10 min warm-start).

In [ ]:
RUN_BRISC = False
if RUN_BRISC:
    load_class('BRISC/DRL/brisc_drl_tumor.yaml', ['brisc2025', 'BRISC'], 'BRISC_tumor')
    full_report('BRISC_tumor')
else:
    print('Skipped (RUN_BRISC = False).')

## 7 · Master summary + per-class recommendation

The actionable bottom line. `realistic_headroom` (attention Dice − baseline Dice) is the honest gain
ceiling; `prob_state` decides whether the uncertainty gate can function; `boundary_frac` decides whether
contour deformation is even the right tool. The recommendation column combines all three.

In [ ]:
def recommend(r):
    rh = r['realistic_headroom']
    if rh is None:
        return 'no attn_dice set — ceiling-only'
    if rh <= 0.005:
        return 'SKIP — no realistic headroom (baseline >= competitor)'
    if r['boundary_frac'] <= 0.5:
        return f'RISKY — only {r["boundary_frac"]:.0%} boundary error; contour can\'t fix the rest'
    gate = ('gate ON (prob_map usable)' if r['prob_state'] == 'USABLE'
            else 'gate OFF or retrain lite w/ label smoothing (prob_map INERT)')
    strength = 'STRONG' if rh > 0.05 else 'GOOD' if rh > 0.02 else 'MARGINAL'
    return f'TRAIN [{strength}, +{rh:.3f} headroom] — {gate}'

rows = list(RESULTS.values())
summary = pd.DataFrame(rows).set_index('label')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
display(summary)
summary.to_csv('/kaggle/working/eda_diagnostic_summary.csv')
print('\nSaved /kaggle/working/eda_diagnostic_summary.csv')

print('\n' + '='*78 + '\nPER-CLASS RECOMMENDATION\n' + '='*78)
# best candidate = largest realistic headroom among boundary-fixable, usable-or-fixable classes
ranked = sorted([r for r in rows if r['realistic_headroom'] is not None],
                key=lambda r: r['realistic_headroom'], reverse=True)
for r in ranked:
    print(f"{r['label']:16s} | baseline {r['baseline_dice']:.4f} -> attn {r['attn_dice']} "
          f"| {r['prob_state']:6s} | boundary {r['boundary_frac']:.2f} | {recommend(r)}")
if ranked:
    print(f"\n>> Best single bet this week: {ranked[0]['label']} "
          f"(+{ranked[0]['realistic_headroom']:.3f} realistic headroom).")